In [1]:
# HW08-09 – PyTorch MLP: регуляризация и оптимизация обучения

import os
import json
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# 1. Seed and device
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# 2. Paths for artifacts
ARTIFACTS_DIR = os.path.join("artifacts")
FIGURES_DIR = os.path.join(ARTIFACTS_DIR, "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)
RUNS_CSV_PATH = os.path.join(ARTIFACTS_DIR, "runs.csv")
BEST_MODEL_PATH = os.path.join(ARTIFACTS_DIR, "best_model.pt")
BEST_CONFIG_PATH = os.path.join(ARTIFACTS_DIR, "best_config.json")


Using device: cpu


In [2]:
# 3. Dataset and DataLoaders (EMNIST balanced)

BATCH_SIZE = 128
VAL_FRACTION = 0.2

transform = transforms.ToTensor()

# Use EMNIST with the "balanced" split
train_full = datasets.EMNIST(root="./data", split="balanced", train=True, download=True, transform=transform)
test_ds = datasets.EMNIST(root="./data", split="balanced", train=False, download=True, transform=transform)

num_train = len(train_full)
num_val = int(num_train * VAL_FRACTION)
num_train_final = num_train - num_val

generator = torch.Generator().manual_seed(SEED)
train_ds, val_ds = random_split(train_full, [num_train_final, num_val], generator=generator)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# Sanity-check a single batch
x_batch, y_batch = next(iter(train_loader))
print("Batch shapes:", x_batch.shape, y_batch.shape)
print("Pixel value range:", float(x_batch.min()), float(x_batch.max()))


Batch shapes: torch.Size([128, 1, 28, 28]) torch.Size([128])
Pixel value range: 0.0 1.0


In [3]:
# 4. MLP model definition

INPUT_DIM = 28 * 28
# Number of classes for EMNIST balanced
NUM_CLASSES = len(train_full.classes)


class MLPBase(nn.Module):
    def __init__(self, hidden_sizes=(256, 128), use_dropout=False, dropout_p=0.5, use_batchnorm=False):
        super().__init__()
        layers: List[nn.Module] = [nn.Flatten()]
        in_dim = INPUT_DIM
        for h in hidden_sizes:
            layers.append(nn.Linear(in_dim, h))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if use_dropout:
                layers.append(nn.Dropout(p=dropout_p))
            in_dim = h
        layers.append(nn.Linear(in_dim, NUM_CLASSES))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


In [4]:
# 5. Training/evaluation utilities

@dataclass
class History:
    train_loss: List[float]
    val_loss: List[float]
    train_acc: List[float]
    val_acc: List[float]


def accuracy_from_logits(logits: torch.Tensor, targets: torch.Tensor) -> float:
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()


def train_one_epoch(model: nn.Module, loader: DataLoader, criterion, optimizer, device: torch.device) -> Tuple[float, float]:
    model.train()
    running_loss = 0.0
    running_acc = 0.0
    n_batches = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        running_acc += accuracy_from_logits(logits, y)
        n_batches += 1

    return running_loss / n_batches, running_acc / n_batches


def evaluate(model: nn.Module, loader: DataLoader, criterion, device: torch.device) -> Tuple[float, float]:
    model.eval()
    running_loss = 0.0
    running_acc = 0.0
    n_batches = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            running_loss += loss.item()
            running_acc += accuracy_from_logits(logits, y)
            n_batches += 1
    return running_loss / n_batches, running_acc / n_batches


In [5]:
# 6. Generic training loop with optional EarlyStopping

@dataclass
class EarlyStoppingConfig:
    patience: int = 3
    mode: str = "max"  # "max" monitors val_accuracy, "min" monitors val_loss


def train_model(
    experiment_id: str,
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion,
    max_epochs: int,
    early_stopping: EarlyStoppingConfig | None = None,
) -> Tuple[History, int, int, float, float]:
    """Train model and keep the best checkpoint.

    Returns:
        history,
        best_epoch (1-based),
        epochs_trained (number of epochs actually run),
        best_val_accuracy,
        best_val_loss.

    Note: On exit, model weights are restored to the best epoch.
    """
    model.to(DEVICE)

    # We always track "best" by val_accuracy for this homework
    best_val_acc = -float("inf")
    best_epoch = 0
    best_state_dict = None

    epochs_no_improve = 0
    history = History(train_loss=[], val_loss=[], train_acc=[], val_acc=[])

    for epoch in range(1, max_epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)

        history.train_loss.append(tr_loss)
        history.val_loss.append(val_loss)
        history.train_acc.append(tr_acc)
        history.val_acc.append(val_acc)

        print(
            f"[{experiment_id}] Epoch {epoch}/{max_epochs} - "
            f"train_loss={tr_loss:.4f}, val_loss={val_loss:.4f}, "
            f"train_acc={tr_acc:.4f}, val_acc={val_acc:.4f}"
        )

        # Track best checkpoint by val_accuracy
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            if early_stopping is not None:
                epochs_no_improve += 1
                if epochs_no_improve >= early_stopping.patience:
                    print(f"Early stopping at epoch {epoch}")
                    break

    epochs_trained = len(history.val_acc)

    # Restore best weights
    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)
        model.to(DEVICE)

    best_val_loss = history.val_loss[best_epoch - 1]
    return history, best_epoch, epochs_trained, best_val_acc, best_val_loss


In [6]:
# 7. Helper to log runs into artifacts/runs.csv

import csv


RUNS_FIELDS = [
    "experiment_id",
    "dataset",
    "seed",
    "model_summary",
    "optimizer",
    "lr",
    "momentum",
    "weight_decay",
    "epochs_trained",
    "best_val_accuracy",
    "best_val_loss",
]


def append_run_row(row: Dict):
    file_exists = os.path.exists(RUNS_CSV_PATH)
    with open(RUNS_CSV_PATH, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=RUNS_FIELDS)
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


In [7]:
# 8. Experiments E1-E4 (regularization)

# Reset runs.csv on each full notebook run to avoid duplicates
if os.path.exists(RUNS_CSV_PATH):
    os.remove(RUNS_CSV_PATH)

MAX_EPOCHS_BASE = 15
hidden_sizes = (256, 128)

criterion = nn.CrossEntropyLoss()

DATASET_NAME = "EMNIST-balanced"


# E1: base MLP, no Dropout/BatchNorm
model_e1 = MLPBase(hidden_sizes=hidden_sizes, use_dropout=False, use_batchnorm=False)
optimizer_e1 = torch.optim.Adam(model_e1.parameters(), lr=1e-3)

hist_e1, best_epoch_e1, epochs_trained_e1, best_val_acc_e1, best_val_loss_e1 = train_model(
    "E1", model_e1, train_loader, val_loader, optimizer_e1, criterion, max_epochs=MAX_EPOCHS_BASE
)

append_run_row(
    {
        "experiment_id": "E1",
        "dataset": DATASET_NAME,
        "seed": SEED,
        "model_summary": f"MLP {hidden_sizes} ReLU, no Dropout, no BatchNorm",
        "optimizer": "Adam",
        "lr": 1e-3,
        "momentum": 0.0,
        "weight_decay": 0.0,
        "epochs_trained": epochs_trained_e1,
        "best_val_accuracy": best_val_acc_e1,
        "best_val_loss": best_val_loss_e1,
    }
)


# E2: Dropout
model_e2 = MLPBase(hidden_sizes=hidden_sizes, use_dropout=True, dropout_p=0.3, use_batchnorm=False)
optimizer_e2 = torch.optim.Adam(model_e2.parameters(), lr=1e-3)

hist_e2, best_epoch_e2, epochs_trained_e2, best_val_acc_e2, best_val_loss_e2 = train_model(
    "E2", model_e2, train_loader, val_loader, optimizer_e2, criterion, max_epochs=MAX_EPOCHS_BASE
)

append_run_row(
    {
        "experiment_id": "E2",
        "dataset": DATASET_NAME,
        "seed": SEED,
        "model_summary": f"MLP {hidden_sizes} ReLU, Dropout p=0.3",
        "optimizer": "Adam",
        "lr": 1e-3,
        "momentum": 0.0,
        "weight_decay": 0.0,
        "epochs_trained": epochs_trained_e2,
        "best_val_accuracy": best_val_acc_e2,
        "best_val_loss": best_val_loss_e2,
    }
)


# E3: BatchNorm
model_e3 = MLPBase(hidden_sizes=hidden_sizes, use_dropout=False, use_batchnorm=True)
optimizer_e3 = torch.optim.Adam(model_e3.parameters(), lr=1e-3)

hist_e3, best_epoch_e3, epochs_trained_e3, best_val_acc_e3, best_val_loss_e3 = train_model(
    "E3", model_e3, train_loader, val_loader, optimizer_e3, criterion, max_epochs=MAX_EPOCHS_BASE
)

append_run_row(
    {
        "experiment_id": "E3",
        "dataset": DATASET_NAME,
        "seed": SEED,
        "model_summary": f"MLP {hidden_sizes} ReLU, BatchNorm",
        "optimizer": "Adam",
        "lr": 1e-3,
        "momentum": 0.0,
        "weight_decay": 0.0,
        "epochs_trained": epochs_trained_e3,
        "best_val_accuracy": best_val_acc_e3,
        "best_val_loss": best_val_loss_e3,
    }
)


# Choose best of E2/E3 by val_accuracy for EarlyStopping model
val_accs = {"E2": best_val_acc_e2, "E3": best_val_acc_e3}
best_regularized_id = max(val_accs, key=val_accs.get)
print("Best regularized experiment before EarlyStopping:", best_regularized_id)

use_dropout_es = best_regularized_id == "E2"
use_batchnorm_es = best_regularized_id == "E3"

model_e4 = MLPBase(hidden_sizes=hidden_sizes, use_dropout=use_dropout_es, dropout_p=0.3, use_batchnorm=use_batchnorm_es)
optimizer_e4 = torch.optim.Adam(model_e4.parameters(), lr=1e-3)

es_cfg = EarlyStoppingConfig(patience=4, mode="max")

hist_e4, best_epoch_e4, epochs_trained_e4, best_val_acc_e4, best_val_loss_e4 = train_model(
    "E4", model_e4, train_loader, val_loader, optimizer_e4, criterion, max_epochs=30, early_stopping=es_cfg
)

append_run_row(
    {
        "experiment_id": "E4",
        "dataset": DATASET_NAME,
        "seed": SEED,
        "model_summary": f"MLP {hidden_sizes} ReLU, Dropout={use_dropout_es}, BatchNorm={use_batchnorm_es}",
        "optimizer": "Adam",
        "lr": 1e-3,
        "momentum": 0.0,
        "weight_decay": 0.0,
        "epochs_trained": epochs_trained_e4,
        "best_val_accuracy": best_val_acc_e4,
        "best_val_loss": best_val_loss_e4,
    }
)


[E1] Epoch 1/15 - train_loss=1.2908, val_loss=0.8298, train_acc=0.6371, val_acc=0.7504
[E1] Epoch 2/15 - train_loss=0.7012, val_loss=0.6561, train_acc=0.7807, val_acc=0.7963
[E1] Epoch 3/15 - train_loss=0.5712, val_loss=0.5841, train_acc=0.8137, val_acc=0.8140
[E1] Epoch 4/15 - train_loss=0.5000, val_loss=0.5381, train_acc=0.8331, val_acc=0.8261
[E1] Epoch 5/15 - train_loss=0.4551, val_loss=0.5185, train_acc=0.8453, val_acc=0.8329
[E1] Epoch 6/15 - train_loss=0.4193, val_loss=0.4945, train_acc=0.8550, val_acc=0.8415
[E1] Epoch 7/15 - train_loss=0.3899, val_loss=0.4958, train_acc=0.8644, val_acc=0.8361
[E1] Epoch 8/15 - train_loss=0.3684, val_loss=0.4998, train_acc=0.8688, val_acc=0.8380
[E1] Epoch 9/15 - train_loss=0.3449, val_loss=0.4938, train_acc=0.8752, val_acc=0.8440
[E1] Epoch 10/15 - train_loss=0.3278, val_loss=0.5077, train_acc=0.8804, val_acc=0.8398
[E1] Epoch 11/15 - train_loss=0.3117, val_loss=0.5047, train_acc=0.8851, val_acc=0.8402
[E1] Epoch 12/15 - train_loss=0.2966, val

In [8]:
# 9. Save best model (E4) and its config, evaluate on test once

# Move best model to device and evaluate on test
model_e4.to(DEVICE)

criterion = nn.CrossEntropyLoss()

test_loss, test_acc = evaluate(model_e4, test_loader, criterion, DEVICE)
print(f"[E4 BEST] Test loss={test_loss:.4f}, test_acc={test_acc:.4f}")

# Save best model state_dict
torch.save(model_e4.state_dict(), BEST_MODEL_PATH)

# Save best config
best_config = {
    "experiment_id": "E4",
    "dataset": "EMNIST-balanced",
    "seed": SEED,
    "hidden_sizes": list(hidden_sizes),
    "use_dropout": use_dropout_es,
    "dropout_p": 0.3 if use_dropout_es else 0.0,
    "use_batchnorm": use_batchnorm_es,
    "optimizer": "Adam",
    "lr": 1e-3,
    "weight_decay": 0.0,
    "batch_size": BATCH_SIZE,
    "max_epochs": 30,
    "early_stopping_patience": 4,
    "test_accuracy": test_acc,
}

with open(BEST_CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(best_config, f, indent=2)

best_config


[E4 BEST] Test loss=0.4771, test_acc=0.8435


{'experiment_id': 'E4',
 'dataset': 'EMNIST-balanced',
 'seed': 42,
 'hidden_sizes': [256, 128],
 'use_dropout': False,
 'dropout_p': 0.0,
 'use_batchnorm': True,
 'optimizer': 'Adam',
 'lr': 0.001,
 'weight_decay': 0.0,
 'batch_size': 128,
 'max_epochs': 30,
 'early_stopping_patience': 4,
 'test_accuracy': 0.8434538995327593}

In [9]:
# 10. Plots for best run (E4) and LR extremes (will be filled after O1/O2)

# Curves for best experiment (E4)
plt.figure(figsize=(8, 4))
plt.plot(hist_e4.train_loss, label="train_loss")
plt.plot(hist_e4.val_loss, label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("E4: Train/Val Loss")
plt.legend()
plt.tight_layout()
curves_best_path = os.path.join(FIGURES_DIR, "curves_best.png")
plt.savefig(curves_best_path)
plt.close()


In [10]:
# 11. Part B: LR diagnostics and SGD+momentum (O1-O3)

# We reuse the same architecture as in E4

criterion = nn.CrossEntropyLoss()


def run_short_experiment(experiment_id: str, optimizer_ctor, max_epochs: int = 8):
    model = MLPBase(hidden_sizes=hidden_sizes, use_dropout=use_dropout_es, dropout_p=0.3, use_batchnorm=use_batchnorm_es)
    optimizer = optimizer_ctor(model.parameters())
    hist, best_epoch, epochs_trained, best_val_acc, best_val_loss = train_model(
        experiment_id, model, train_loader, val_loader, optimizer, criterion, max_epochs=max_epochs
    )
    append_run_row(
        {
            "experiment_id": experiment_id,
            "dataset": "EMNIST-balanced",
            "seed": SEED,
            "model_summary": f"MLP {hidden_sizes} ReLU, Dropout={use_dropout_es}, BatchNorm={use_batchnorm_es}",
            "optimizer": optimizer.__class__.__name__,
            "lr": optimizer.param_groups[0]["lr"],
            "momentum": optimizer.param_groups[0].get("momentum", 0.0),
            "weight_decay": optimizer.param_groups[0].get("weight_decay", 0.0),
            "epochs_trained": epochs_trained,
            "best_val_accuracy": best_val_acc,
            "best_val_loss": best_val_loss,
        }
    )
    return hist


# O1: LR too large (Adam)
hist_o1 = run_short_experiment(
    "O1",
    optimizer_ctor=lambda params: torch.optim.Adam(params, lr=1e-1),
    max_epochs=8,
)

# O2: LR too small (Adam)
hist_o2 = run_short_experiment(
    "O2",
    optimizer_ctor=lambda params: torch.optim.Adam(params, lr=1e-5),
    max_epochs=8,
)

# O3: SGD + momentum + weight decay
hist_o3 = run_short_experiment(
    "O3",
    optimizer_ctor=lambda params: torch.optim.SGD(params, lr=5e-3, momentum=0.9, weight_decay=1e-4),
    max_epochs=12,
)


[O1] Epoch 1/8 - train_loss=1.0868, val_loss=0.8206, train_acc=0.6702, val_acc=0.7428
[O1] Epoch 2/8 - train_loss=0.7794, val_loss=0.7868, train_acc=0.7512, val_acc=0.7567
[O1] Epoch 3/8 - train_loss=0.7016, val_loss=0.6658, train_acc=0.7726, val_acc=0.7915
[O1] Epoch 4/8 - train_loss=0.6588, val_loss=0.6994, train_acc=0.7854, val_acc=0.7876
[O1] Epoch 5/8 - train_loss=0.6270, val_loss=0.6468, train_acc=0.7942, val_acc=0.7869
[O1] Epoch 6/8 - train_loss=0.6020, val_loss=0.6324, train_acc=0.8002, val_acc=0.8056
[O1] Epoch 7/8 - train_loss=0.5847, val_loss=0.6079, train_acc=0.8051, val_acc=0.8119
[O1] Epoch 8/8 - train_loss=0.5768, val_loss=0.6152, train_acc=0.8069, val_acc=0.8080
[O2] Epoch 1/8 - train_loss=3.4652, val_loss=3.1388, train_acc=0.1785, val_acc=0.3329
[O2] Epoch 2/8 - train_loss=2.9314, val_loss=2.7549, train_acc=0.4179, val_acc=0.4776
[O2] Epoch 3/8 - train_loss=2.6111, val_loss=2.4787, train_acc=0.5151, val_acc=0.5429
[O2] Epoch 4/8 - train_loss=2.3657, val_loss=2.2561, t

In [11]:
# 12. Plots for LR extremes (O1, O2)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(hist_o1.train_loss, label="O1 train")
plt.plot(hist_o1.val_loss, label="O1 val")
plt.title("O1: LR too large (Adam, lr=1e-1)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(hist_o2.train_loss, label="O2 train")
plt.plot(hist_o2.val_loss, label="O2 val")
plt.title("O2: LR too small (Adam, lr=1e-5)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
curves_lr_extremes_path = os.path.join(FIGURES_DIR, "curves_lr_extremes.png")
plt.savefig(curves_lr_extremes_path)
plt.close()

curves_best_path, curves_lr_extremes_path


('artifacts\\figures\\curves_best.png',
 'artifacts\\figures\\curves_lr_extremes.png')